# Preprocessing Quality Control

Step through the full preprocessing pipeline: bandpass filtering, notch filtering,
z-score normalization, and downsampling. At each stage, visualize the signal in both
time and frequency domains to verify correctness.

## 1. Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal
%matplotlib inline

from src.config import get_default_config
from src.data.loader import load_willett_dataset
from src.preprocessing.filter import bandpass_filter, notch_filter
from src.preprocessing.normalize import zscore_normalize
from src.diagnostics.spectral_analysis import compute_psd

## 2. Load Data

In [ ]:
cfg = get_default_config()
trial_index = load_willett_dataset(cfg)

# Load a representative trial
sample_row = trial_index.iloc[0]
raw_signal = np.load(sample_row['signal_path'])
fs = cfg.target_fs
print(f"Raw signal shape: {raw_signal.shape}  (timesteps x channels)")
print(f"Sampling rate: {fs} Hz")

## Helper: Plot Before/After Spectra

Utility to show the effect of each preprocessing step in both time and frequency domains.

In [ ]:
def plot_before_after(before, after, fs, title, channel=0):
    """Plot a single channel before and after a processing step."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 6))

    # Time domain
    t = np.arange(before.shape[0]) / fs
    axes[0, 0].plot(t, before[:, channel], linewidth=0.5)
    axes[0, 0].set_title(f'Before — Ch {channel} (time)')
    axes[0, 0].set_xlabel('Time (s)')

    axes[0, 1].plot(t[:after.shape[0]], after[:, channel], linewidth=0.5, color='C1')
    axes[0, 1].set_title(f'After — Ch {channel} (time)')
    axes[0, 1].set_xlabel('Time (s)')

    # Frequency domain (Welch PSD)
    nperseg = min(256, before.shape[0])
    f_b, psd_b = scipy_signal.welch(before[:, channel], fs=fs, nperseg=nperseg)
    f_a, psd_a = scipy_signal.welch(after[:, channel], fs=fs, nperseg=min(256, after.shape[0]))

    axes[1, 0].semilogy(f_b, psd_b)
    axes[1, 0].set_title('Before — PSD')
    axes[1, 0].set_xlabel('Frequency (Hz)')

    axes[1, 1].semilogy(f_a, psd_a, color='C1')
    axes[1, 1].set_title('After — PSD')
    axes[1, 1].set_xlabel('Frequency (Hz)')

    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    return fig

## 3. Apply Bandpass Filter

Remove very low-frequency drift and high-frequency noise outside the neural band of interest.

In [ ]:
bandpassed = bandpass_filter(raw_signal, fs=fs)
print(f"After bandpass: shape={bandpassed.shape}")
plot_before_after(raw_signal, bandpassed, fs, 'Bandpass Filtering')

## 4. Apply Notch Filter

Remove 60 Hz power-line interference (and harmonics if present).

In [ ]:
notched = notch_filter(bandpassed, fs=fs)
print(f"After notch filter: shape={notched.shape}")
plot_before_after(bandpassed, notched, fs, 'Notch Filtering (60 Hz)')

## 5. Z-Score Normalization

Normalize each channel to zero mean and unit variance for stable model training.

In [ ]:
normalized = zscore_normalize(notched)
print(f"After normalization: shape={normalized.shape}")
print(f"Channel means (first 10): {normalized.mean(axis=0)[:10].round(6)}")
print(f"Channel stds  (first 10): {normalized.std(axis=0)[:10].round(6)}")

plot_before_after(notched, normalized, fs, 'Z-Score Normalization')

## 6. Downsampling

Reduce the sampling rate by an integer factor to speed up model training
while preserving the frequencies of interest.

In [ ]:
downsample_factor = 2
downsampled = normalized[::downsample_factor, :]
new_fs = fs / downsample_factor
print(f"After downsampling by {downsample_factor}x: shape={downsampled.shape}")
print(f"New effective sampling rate: {new_fs} Hz")

plot_before_after(normalized, downsampled, fs, f'Downsampling ({downsample_factor}x)')

## 7. Final Preprocessed Signal

Show the fully preprocessed signal and compare to the raw input.

In [ ]:
final_signal = downsampled

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=False)

# Raw
t_raw = np.arange(raw_signal.shape[0]) / fs
for ch in range(min(5, raw_signal.shape[1])):
    axes[0].plot(t_raw, raw_signal[:, ch], linewidth=0.3, alpha=0.7)
axes[0].set_title('Raw Signal (first 5 channels)')
axes[0].set_xlabel('Time (s)')

# Preprocessed
t_proc = np.arange(final_signal.shape[0]) / new_fs
for ch in range(min(5, final_signal.shape[1])):
    axes[1].plot(t_proc, final_signal[:, ch], linewidth=0.3, alpha=0.7)
axes[1].set_title('Preprocessed Signal (first 5 channels)')
axes[1].set_xlabel('Time (s)')

plt.tight_layout()
plt.show()

print(f"\nPreprocessing summary:")
print(f"  Raw shape:          {raw_signal.shape}")
print(f"  Preprocessed shape: {final_signal.shape}")
print(f"  Original fs:        {fs} Hz")
print(f"  Final fs:           {new_fs} Hz")

---

**Next:** Proceed to `04_model_training.ipynb` for model training and comparison.